# Session 9 Solutions: Advanced PySpark

This Colab notebook runs the Session 9 solution path end to end.

It covers:

- loading a CSV with Spark
- checking schema, rows, and columns
- registering SQL views
- creating derived columns and time features
- running Spark SQL analytics
- building ranked service summaries with window functions
- saving `results/service_summary.csv`
- running simple checks to confirm the workflow worked


## 1. Install PySpark if needed

Run this first in Google Colab. If PySpark is already installed, the cell will skip installation.

In [ ]:
try:
    import pyspark
    print("PySpark already available:", pyspark.__version__)
except ModuleNotFoundError:
    print("Installing PySpark...")
    %pip install pyspark==4.1.2


## 2. Start Spark

In [ ]:
from pathlib import Path
import shutil

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    col,
    count,
    date_format,
    dense_rank,
    hour,
    round,
    stddev,
    sum,
    to_date,
    when,
)
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Session09SolutionsColab")
    .master("local[*]")
    .getOrCreate()
)

spark


## 3. Create the tutorial CSV inside Colab

This makes the notebook self-contained. Students can also upload `datasets/service_events.csv`, but this cell is enough for testing.

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
events_path = data_dir / "service_events.csv"

events_csv = """event_id,service,region,event_time,request_count,error_count,latency_ms,bytes_in,bytes_out
1,auth,eu-west,2026-05-04 00:00:00,1240,8,82.4,14500000,38200000
2,payments,eu-west,2026-05-04 00:00:00,860,21,146.2,19200000,44100000
3,search,us-east,2026-05-04 01:00:00,2110,13,94.8,28700000,76300000
4,recommendations,us-east,2026-05-04 01:00:00,1680,18,132.5,34800000,91100000
5,auth,ap-south,2026-05-04 02:00:00,980,4,78.1,11600000,30100000
6,payments,ap-south,2026-05-04 02:00:00,740,17,171.3,18500000,42600000
7,search,eu-west,2026-05-04 03:00:00,2320,15,101.7,30900000,84200000
8,recommendations,eu-west,2026-05-04 03:00:00,1760,24,155.9,36600000,98400000
9,auth,us-east,2026-05-04 09:00:00,1480,6,74.6,17100000,45200000
10,payments,us-east,2026-05-04 09:00:00,1120,26,189.4,24800000,59900000
11,search,ap-south,2026-05-04 10:00:00,1980,11,88.9,26300000,70100000
12,recommendations,ap-south,2026-05-04 10:00:00,1420,16,128.8,30400000,82700000
13,auth,eu-west,2026-05-05 00:00:00,1310,7,80.2,15100000,39900000
14,payments,eu-west,2026-05-05 00:00:00,910,19,151.5,20100000,46600000
15,search,us-east,2026-05-05 01:00:00,2200,12,92.1,29500000,79000000
16,recommendations,us-east,2026-05-05 01:00:00,1710,20,141.0,35400000,94900000
17,auth,ap-south,2026-05-05 02:00:00,1010,5,76.5,11900000,31800000
18,payments,ap-south,2026-05-05 02:00:00,780,22,183.6,19100000,45100000
19,search,eu-west,2026-05-05 03:00:00,2410,14,99.2,32100000,87500000
20,recommendations,eu-west,2026-05-05 03:00:00,1820,27,164.7,37800000,100200000
21,auth,us-east,2026-05-05 09:00:00,1560,9,86.0,17900000,47100000
22,payments,us-east,2026-05-05 09:00:00,1180,31,205.8,25600000,62100000
23,search,ap-south,2026-05-05 10:00:00,2060,10,87.4,27100000,72400000
24,recommendations,ap-south,2026-05-05 10:00:00,1490,14,121.6,31300000,85200000
"""

events_path.write_text(events_csv, encoding="utf-8")
print(f"Created {events_path}")


## 4. Part 1 solution: load the CSV, inspect it, and create a SQL view

In [ ]:
schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

events_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(str(events_path))
)

events_df.printSchema()
events_df.show(5, truncate=False)

print("Rows:", events_df.count())
print("Columns:", events_df.columns)

events_df.createOrReplaceTempView("service_events")


In [ ]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM service_events
""").show()

spark.sql("""
    SELECT DISTINCT service
    FROM service_events
    ORDER BY service
""").show()

spark.sql("""
    SELECT region, COUNT(*) AS row_count
    FROM service_events
    GROUP BY region
    ORDER BY row_count DESC
""").show()


## 5. Part 2 solution: derived columns and time features

In [ ]:
events_enriched_df = (
    events_df
    .withColumn(
        "error_rate",
        when(col("request_count") > 0, col("error_count") / col("request_count")).otherwise(0),
    )
    .withColumn("total_bytes", col("bytes_in") + col("bytes_out"))
    .withColumn("traffic_mb", col("total_bytes") / 1048576)
    .withColumn(
        "latency_band",
        when(col("latency_ms") < 100, "fast")
        .when(col("latency_ms") < 160, "normal")
        .otherwise("slow"),
    )
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("day_of_week", date_format(col("event_time"), "E"))
)

events_enriched_df.select(
    "service",
    "event_time",
    "event_date",
    "event_hour",
    "day_of_week",
    round("error_rate", 4).alias("error_rate"),
    round("traffic_mb", 2).alias("traffic_mb"),
    "latency_band",
).show(10, truncate=False)

events_enriched_df.createOrReplaceTempView("service_events_enriched")


In [ ]:
spark.sql("""
    SELECT
        service,
        ROUND(AVG(error_rate), 4) AS average_error_rate
    FROM service_events_enriched
    GROUP BY service
    ORDER BY average_error_rate DESC
""").show()

spark.sql("""
    SELECT
        service,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms
    FROM service_events_enriched
    GROUP BY service
    ORDER BY average_latency_ms DESC
""").show()

spark.sql("""
    SELECT
        event_hour,
        SUM(request_count) AS total_requests
    FROM service_events_enriched
    GROUP BY event_hour
    ORDER BY total_requests DESC
""").show()

spark.sql("""
    SELECT
        region,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY region
    ORDER BY total_traffic_mb DESC
""").show()

spark.sql("""
    SELECT latency_band, COUNT(*) AS row_count
    FROM service_events_enriched
    GROUP BY latency_band
    ORDER BY row_count DESC
""").show()

spark.sql("""
    SELECT
        event_date,
        SUM(request_count) AS total_requests,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY event_date
    ORDER BY event_date
""").show()


## 6. Part 3 solution: rankings and final summary

In [ ]:
service_summary_df = events_enriched_df.groupBy("service").agg(
    count("*").alias("total_records"),
    sum("request_count").alias("total_requests"),
    sum("error_count").alias("total_errors"),
    avg("error_rate").alias("average_error_rate"),
    avg("latency_ms").alias("average_latency_ms"),
    stddev("latency_ms").alias("latency_stddev"),
    sum("traffic_mb").alias("total_traffic_mb"),
)

traffic_window = Window.orderBy(col("total_traffic_mb").desc())
variability_window = Window.orderBy(col("latency_stddev").desc())
reliability_window = Window.orderBy(col("average_error_rate").asc())

ranked_summary_df = (
    service_summary_df
    .withColumn("traffic_rank", dense_rank().over(traffic_window))
    .withColumn("latency_variability_rank", dense_rank().over(variability_window))
    .withColumn("reliability_rank", dense_rank().over(reliability_window))
)

ranked_summary_df.orderBy("traffic_rank").show(truncate=False)


In [ ]:
hourly_requests_df = events_enriched_df.groupBy("service", "event_hour").agg(
    sum("request_count").alias("hourly_requests")
)

hour_window = Window.partitionBy("service").orderBy(col("hourly_requests").desc())

busiest_hour_df = (
    hourly_requests_df
    .withColumn("hour_rank", dense_rank().over(hour_window))
    .filter(col("hour_rank") == 1)
    .select("service", col("event_hour").alias("busiest_hour"))
)

busiest_hour_df.show()


In [ ]:
final_summary_df = (
    ranked_summary_df
    .join(busiest_hour_df, on="service", how="left")
    .select(
        "service",
        "total_records",
        "total_requests",
        "total_errors",
        round("average_error_rate", 4).alias("average_error_rate"),
        round("average_latency_ms", 2).alias("average_latency_ms"),
        round("latency_stddev", 2).alias("latency_stddev"),
        round("total_traffic_mb", 2).alias("total_traffic_mb"),
        "busiest_hour",
        "traffic_rank",
        "latency_variability_rank",
        "reliability_rank",
    )
    .orderBy("traffic_rank")
)

final_summary_df.show(truncate=False)


## 7. Save `results/service_summary.csv`

In [ ]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

spark_output_folder = results_dir / "service_summary_spark"
single_csv = results_dir / "service_summary.csv"

if spark_output_folder.exists():
    shutil.rmtree(spark_output_folder)

final_summary_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(spark_output_folder))

if single_csv.exists():
    single_csv.unlink()

part_file = next(spark_output_folder.glob("part-*.csv"))
shutil.copy(part_file, single_csv)

print(f"Saved {single_csv}")


## 8. Verify everything worked

In [ ]:
assert events_df.count() == 24
assert set(["error_rate", "traffic_mb", "latency_band", "event_hour"]).issubset(set(events_enriched_df.columns))
assert final_summary_df.count() == events_df.select("service").distinct().count()
assert single_csv.exists()

saved_df = spark.read.option("header", True).csv(str(single_csv))
saved_df.show(truncate=False)

top_traffic = final_summary_df.orderBy("traffic_rank").first()
top_variability = final_summary_df.orderBy("latency_variability_rank").first()
top_reliability = final_summary_df.orderBy("reliability_rank").first()

print("Session 9 checks passed.")
print(f"Top traffic service: {top_traffic['service']}")
print(f"Most variable latency service: {top_variability['service']}")
print(f"Most reliable service: {top_reliability['service']}")


## 9. Stop Spark

In [ ]:
spark.stop()
print("Spark stopped")
